<a href="https://colab.research.google.com/github/shanusushmita/CS4973-Applied-Multilingual-Systems/blob/main/Code_Switch_Metrics_(CSI%2C_MUR%2C_CSE).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import math
from collections import Counter

# -----------------------------------------
# Mock bilingual dialogue: Speaker A & B
# -----------------------------------------
dialogue = [
    {"speaker": "A", "text": "Hey, cómo estás today?"},
    {"speaker": "B", "text": "Estoy bien, just a little tired."},
    {"speaker": "A", "text": "Yeah, same here. No dormí mucho anoche."},
    {"speaker": "B", "text": "We should tomar un café before class."},
    {"speaker": "A", "text": "Good idea! Necesito caffeine urgently."},
    {"speaker": "B", "text": "Perfecto, let's meet en la cafetería."},
    {"speaker": "A", "text": "Okay, see you pronto."},
    {"speaker": "B", "text": "Hasta luego!"}
]

# -----------------------------------------
# Simplified token-level language tags (mock)
# -----------------------------------------
utterances_labels = [
    ["EN", "ES", "ES", "EN"],                    # A
    ["ES", "ES", "EN", "EN", "EN"],              # B
    ["EN", "EN", "EN", "ES", "ES", "ES"],        # A
    ["EN", "EN", "ES", "ES", "ES"],              # B
    ["EN", "EN", "ES", "EN"],                    # A
    ["ES", "EN", "EN", "ES", "ES"],              # B
    ["EN", "EN", "ES"],                          # A
    ["ES", "ES"]                                 # B
]

# Add labels back to dialogue
for d, labels in zip(dialogue, utterances_labels):
    d["labels"] = labels

# -----------------------------------------
# Helper functions
# -----------------------------------------
def compute_MUR(utterances):
    """Proportion of utterances containing more than one language."""
    total = len(utterances)
    mixed = sum(len(set(utt["labels"])) > 1 for utt in utterances)
    return mixed / total

def compute_MUR_per_speaker(dialogue):
    """Compute MUR for each speaker separately."""
    speakers = {}
    for speaker in set(d["speaker"] for d in dialogue):
        utts = [d for d in dialogue if d["speaker"] == speaker]
        speakers[speaker] = compute_MUR(utts)
    return speakers

def compute_CSI(dialogue):
    """Token-level switching frequency."""
    total_tokens = sum(len(d["labels"]) for d in dialogue)
    switch_points = 0
    for d in dialogue:
        for i in range(1, len(d["labels"])):
            if d["labels"][i] != d["labels"][i - 1]:
                switch_points += 1
    return switch_points / total_tokens

def compute_CSE(dialogue):
    """Code-Switching Entropy based on overall language distribution."""
    all_tokens = [lang for d in dialogue for lang in d["labels"]]
    counts = Counter(all_tokens)
    total = sum(counts.values())
    proportions = [c / total for c in counts.values()]
    return -sum(p * math.log2(p) for p in proportions)

# -----------------------------------------
# Compute metrics
# -----------------------------------------
MUR_overall = compute_MUR(dialogue)
MUR_speakers = compute_MUR_per_speaker(dialogue)
CSI = compute_CSI(dialogue)
CSE = compute_CSE(dialogue)

# -----------------------------------------
# Display results
# -----------------------------------------
print("=== Dialogue Sample ===\n")
for d in dialogue:
    print(f"{d['speaker']}: {d['text']}")
    print(f"   Labels: {d['labels']}\n")

print("=== Code-Switching Metrics ===")
print(f"MUR (Overall): {MUR_overall:.2f}")
for spk, val in MUR_speakers.items():
    print(f"  MUR ({spk}): {val:.2f}")
print(f"CSI (Code-Switching Index): {CSI:.2f}")
print(f"CSE (Code-Switching Entropy): {CSE:.2f}")


=== Dialogue Sample ===

A: Hey, cómo estás today?
   Labels: ['EN', 'ES', 'ES', 'EN']

B: Estoy bien, just a little tired.
   Labels: ['ES', 'ES', 'EN', 'EN', 'EN']

A: Yeah, same here. No dormí mucho anoche.
   Labels: ['EN', 'EN', 'EN', 'ES', 'ES', 'ES']

B: We should tomar un café before class.
   Labels: ['EN', 'EN', 'ES', 'ES', 'ES']

A: Good idea! Necesito caffeine urgently.
   Labels: ['EN', 'EN', 'ES', 'EN']

B: Perfecto, let's meet en la cafetería.
   Labels: ['ES', 'EN', 'EN', 'ES', 'ES']

A: Okay, see you pronto.
   Labels: ['EN', 'EN', 'ES']

B: Hasta luego!
   Labels: ['ES', 'ES']

=== Code-Switching Metrics ===
MUR (Overall): 0.88
  MUR (B): 0.75
  MUR (A): 1.00
CSI (Code-Switching Index): 0.29
CSE (Code-Switching Entropy): 1.00
